# 03 — Baseline Model: Linear Regression

**Goal (game plan Week 4):** train a Linear Regression as the first model and record baseline results.
**Author:** Ethan Chung — IDX Exchange Data Science, Fall 2026

Following the IDX Best Practices doc:
- **Target:** `log(ClosePrice)`; every metric is computed back in dollars.
- **Window selection on validation only:** fit on 3 / 6 / 12 / 24 months before the 2026-03 validation month, keep the lowest validation MdAPE. The test month is not looked at.
- **Test once:** refit with the chosen window on the months right before 2026-04 and score 2026-04.
- **Rolling backtest:** same window, test months 2026-03, 2026-02, 2026-01.
- **Metrics:** R², MAPE, MdAPE (headline), MAE, RMSE, plus error by price band and county.
- **Leakage-safe:** the model is `Pipeline(preprocessor, LinearRegression)`, so all preprocessing is fit on training rows only.

Logic lives in `src/models.py` and `src/evaluation.py`; run everything with `python src/models.py`.

## 1. Setup

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

sys.path.insert(0, os.path.abspath("../src"))
import evaluation
import models
import preprocessing

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
METRICS = ["n", "R2", "MAPE", "MdAPE", "MAE", "RMSE", "within_10pct", "median_pct_error"]

In [ ]:
clean = preprocessing.load_clean("../data/processed/sfr_clean.parquet")
result = models.run("baseline_linear_regression", LinearRegression, df=clean, feature_set="week5")

## 2. Training-window selection (validation month 2026-03)

In [ ]:
result["selection"][["train_months", "train_start", "train_end", "train_rows"] + METRICS]

In [ ]:
print("Chosen window:", result["best_window"], "months")

Validation MdAPE improves only slightly with more history (11.8% for 3 months → 11.1% for 24 months).
For a linear model the window length matters little; the 24-month window wins on every percentage metric, so it is used.

## 3. Test month 2026-04 (scored once)

In [ ]:
test_metrics = pd.Series(result["test"])
test_metrics

## 4. Rolling-origin backtest

In [ ]:
backtest = result["backtest"][["test_month", "train_start", "train_end"] + METRICS]
backtest

In [ ]:
stability = pd.concat([pd.DataFrame([result["test"]]), result["backtest"]])[["test_month", "R2", "MdAPE", "MAPE"]]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
for ax, col in zip(axes, ["R2", "MdAPE"]):
    s = stability.sort_values("test_month")
    ax.plot(s["test_month"], s[col], marker="o")
    ax.set_title(f"{col} by test month (24-month window)")
axes[0].set_ylim(0.7, 0.9); axes[1].set_ylim(0, 15)
plt.tight_layout(); plt.show()

## 5. Error breakdown

In [ ]:
bands = result["by_band"][["group", "n", "MdAPE", "MAPE", "MAE", "within_10pct", "median_pct_error"]]
bands

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].bar(bands["group"].str.split(" ").str[0], bands["MdAPE"])
axes[0].set_title("MdAPE by price quintile (%)")
axes[1].bar(bands["group"].str.split(" ").str[0], bands["median_pct_error"])
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title("Median signed error by price quintile (%)  (> 0 = over-valued)")
plt.tight_layout(); plt.show()

In [ ]:
result["by_county"].sort_values("MdAPE")[["group", "n", "MdAPE", "MAPE", "MAE", "median_pct_error"]]

In [ ]:
test_rows, pred = result["test_rows"], result["pred"]
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.hexbin(test_rows["ClosePrice"], pred, gridsize=60, xscale="log", yscale="log", mincnt=1, bins="log")
lims = [test_rows["ClosePrice"].min(), test_rows["ClosePrice"].max()]
ax.plot(lims, lims, color="black", linewidth=1)
ax.set_xlabel("actual close price ($)"); ax.set_ylabel("predicted ($)")
ax.set_title("Predicted vs actual, test 2026-04")
plt.tight_layout(); plt.show()

## 6. What the model learned

In [ ]:
import features

lr = result["model"]
coefs = pd.Series(lr.named_steps["model"].coef_, index=lr.named_steps["preprocess"].get_feature_names_out())
# Only standardized inputs are comparable: with a log target, 0.10 ≈ +10% price per 1 SD of the feature.
# County / Levels dummies and missing flags are 0/1 and overlap with the ZIP and city encodings, so they are left out.
week5 = features.FEATURE_SETS["week5"]
standardized = week5["log_numeric"] + week5["numeric"] + week5["high_cardinality"]
top = coefs[standardized].reindex(coefs[standardized].abs().sort_values(ascending=False).index)
top.to_frame("coefficient (log price per SD)")

## 7. Findings

- **Baseline:** test R² **0.82**, MdAPE **11.2%**, MAPE 15.4%, MAE ≈ $209k. 45% of homes are valued within 10%.
- **Stable over time:** R² 0.79–0.82 and MdAPE 11.1–11.4% across the four test months, so the result is not a lucky cutoff.
- **Regression to the mean:** the model over-values entry-level homes (median +4.9% in the lowest quintile) and under-values luxury homes (median −10.6% and MdAPE 15.9% in the top quintile). A linear model in log price cannot bend enough at the extremes.
- **Drivers:** location (ZIP target encoding, latitude/longitude) and living area dominate. Bedrooms are slightly negative once size is known: more rooms in the same square footage means smaller rooms.
- **Geography:** error is lowest in the Inland Empire (San Bernardino 9.4%, Riverside 10.5%) and highest in the Bay Area counties (Alameda, Santa Clara ≈ 14%), where prices vary more within a ZIP.

**Limitations:** one linear relationship statewide; no interactions (e.g. size × location); predictions are `exp` of a log-scale prediction, which estimates the median price rather than the mean.

**Next (Week 5):** Decision Tree and Random Forest with the same pipeline, compared side by side with this baseline (`04_model_comparison.ipynb`).